# VAD Detection with Silero VAD

Run Silero VAD on WAV files to detect speech regions.
Uses low threshold (0.1) to catch soft fillers (呃/嗯/啊).
Outputs `[start_ms, end_ms]` segments per file.

In [ ]:
from pathlib import Path

WAV_DIR = Path("../data/wav")
OUTPUT_PATH = Path("../data/vad_results.jsonl")

# Silero VAD parameters — per PodcastFillers paper, 0.1 threshold is critical
# to catch soft fillers without missing them
THRESHOLD = 0.1
MIN_SPEECH_MS = 150     # merge speech chunks shorter than this
MIN_SILENCE_MS = 100    # don't split on silences shorter than this

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
)

(get_speech_timestamps, _, read_audio, _, _) = utils

# Move to MPS if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print(f"Silero VAD loaded on {device}")

In [ ]:
wav_files = sorted(WAV_DIR.glob("*.wav"))
print(f"Found {len(wav_files)} WAV files in {WAV_DIR}")

In [ ]:
import json

SAMPLE_RATE = 16000

for i, wav_path in enumerate(wav_files):
    wav = read_audio(str(wav_path), sampling_rate=SAMPLE_RATE).to(device)

    timestamps = get_speech_timestamps(
        wav,
        model,
        threshold=THRESHOLD,
        min_speech_duration_ms=MIN_SPEECH_MS,
        min_silence_duration_ms=MIN_SILENCE_MS,
        sampling_rate=SAMPLE_RATE,
        return_seconds=False,
    )

    # Convert sample indices to [start_ms, end_ms]
    segments = [
        [int(t["start"] / SAMPLE_RATE * 1000), int(t["end"] / SAMPLE_RATE * 1000)]
        for t in timestamps
    ]

    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        record = {"file": wav_path.name, "segments": segments}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    if (i + 1) % 10 == 0 or (i + 1) == len(wav_files):
        print(f"[{i + 1}/{len(wav_files)}] {wav_path.name} — {len(segments)} segments")

print(f"\nDone. Results saved to {OUTPUT_PATH}")

In [ ]:
# Preview results
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        segs = r["segments"]
        print(f"--- {r['file']} ---")
        print(f"  {len(segs)} speech segments")
        for s in segs[:5]:
            print(f"  {s[0]:>8}–{s[1]:>8} ms  ({s[1] - s[0]} ms)")
        if len(segs) > 5:
            print(f"  ... and {len(segs) - 5} more")
        print()